# Python for Cybersecurity Analysis

Bu notebook yalnızca dərslik üçün yaradılmış sintetik `security_logs.csv` faylını təhlil edir. Real şəbəkəyə qoşulmur və hücum yaratmır.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# CSV faylını notebook ilə eyni qovluqda saxlayın
df = pd.read_csv('security_logs.csv')
df.head()

## 1. Məlumatların ilkin yoxlanması
`shape` sətir və sütun sayını, `info()` tipləri, `isna()` isə boş xanaları göstərir.

In [ ]:
print('Ölçü:', df.shape)
print(df.info())
print('Boş xanalar:\n', df.isna().sum())
df['timestamp'] = pd.to_datetime(df['timestamp'])

## 2. Uğursuz giriş cəhdlərinin seçilməsi
`LOGIN_FAILURE` hadisələri uğursuz autentifikasiya cəhdlərini ifadə edən sintetik qeydlərdir.

In [ ]:
failed = df[df['event_type'] == 'LOGIN_FAILURE']
print('Uğursuz giriş sayı:', len(failed))
failed_by_ip = failed.groupby('ip_address').size().sort_values(ascending=False)
failed_by_ip

## 3. IP ünvanlarına görə qruplaşdırma
`groupby` eyni IP ünvanına aid qeydləri bir qrupda birləşdirir.

In [ ]:
ip_summary = df.groupby('ip_address').agg(
    event_count=('event_type', 'count'),
    failed_logins=('status', lambda s: (s == 'FAILED').sum()),
    review_events=('status', lambda s: (s == 'REVIEW').sum())
).sort_values('event_count', ascending=False)
ip_summary

## 4. Port və hadisə təhlili
Port 4444 bu nümunə datasetində araşdırma üçün işarələnmiş qeyri-adi portdur; bu, öz-özlüyündə zərərli fəaliyyətin sübutu deyil.

In [ ]:
print(df.groupby(['port', 'event_type']).size().sort_values(ascending=False).head(12))
review = df[df['status'] == 'REVIEW']
review[['timestamp','ip_address','user','event_type','port','status']].head(10)

## 5. Hadisələrin zaman üzrə vizuallaşdırılması

In [ ]:
hourly = df.set_index('timestamp').resample('1h').size()
ax = hourly.plot(kind='line', marker='o', title='Saatlıq sintetik hadisə sayı')
ax.set_xlabel('Vaxt'); ax.set_ylabel('Hadisə sayı'); plt.tight_layout(); plt.show()

## 6. Sadə anomaliya aşkarlanması (statistik qayda)
Burada hər IP üzrə uğursuz giriş sayı hesablanır. Hədd tədris məqsədi ilə seçilib; istehsalat sistemində kontekst və baza xətti tələb olunur.

In [ ]:
counts = failed.groupby('ip_address').size()
threshold = max(3, int(counts.mean() + 2 * counts.std(ddof=0))) if len(counts) else 3
anomalies = counts[counts >= threshold]
print('Tədris həddi:', threshold)
print('Araşdırılmalı IP-lər:\n', anomalies)

## 7. Scikit-learn ilə Isolation Forest
Model yalnız sintetik nümunənin sadə say göstəricilərini öyrənir. Nəticə real hücum təsdiqi deyil.

In [ ]:
try:
    from sklearn.ensemble import IsolationForest
    features = df.groupby('ip_address').agg(
        events=('event_type','count'),
        failed=('status', lambda s: (s == 'FAILED').sum()),
        review=('status', lambda s: (s == 'REVIEW').sum())
    )
    model = IsolationForest(contamination=0.2, random_state=8)
    features['model_flag'] = model.fit_predict(features)
    features['model_flag'] = features['model_flag'].map({1:'Normal nümunə', -1:'Anomaliya namizədi'})
    display(features.sort_values('failed', ascending=False))
except ImportError:
    print('Scikit-learn quraşdırılmayıb: pip install scikit-learn')

## 8. Nəticələrin şərhi
- Modelin işarələdiyi qeydlər araşdırma üçün namizəddir, avtomatik hökm deyil.
- Yanlış müsbət (false positive) və yanlış mənfi (false negative) nəticələr mümkündür.
- Hadisəni təsdiqləmək üçün aktiv, istifadəçi, vaxt, şəbəkə konteksti və əlavə loglar yoxlanmalıdır.